# Vachan V2 — a *per-persona* tone dial (control vectors)

**What this adds over the spike:** the first notebook proved the *mechanism* — you can move tone formal-English ↔ Hinglish by adding one direction to the hidden states. This one makes that direction **person-specific**: instead of a generic "be Hinglish" instruction, we build the contrast from **one real person's own messages** vs their plain-English translation. The dial then reproduces *that individual's voice*, not a generic register.

**Where the data comes from:** in Vachan every persona capsule already stores both sides — `anchors` (the person's real messages) and `anchors_english` (their English translations). Here we paste a handful by hand so the notebook stays standalone; in production you pull them from the capsule.

**n8n analogy:** same slider node as before, but now the slider is *calibrated to one person* — like a workflow variable filled from that contact's record instead of a hardcoded default.

> Runtime ~5-10 min on a Kaggle **T4**. Reuses the hardened setup from the spike (numpy repair, fixed generate) — Run All, no manual steps.

## 0. Setup (read me)

1. **Kaggle**: top-right **⋮ → Accelerator → GPU T4 x2** (or just T4). Also **Internet: On** (Settings).
2. **Model**: non-gated Llama-3.1-8B mirror — no HuggingFace token needed.
3. Run cells top to bottom (**Run All**).
4. **Two gotchas, both normal:** the install cell prints red "dependency conflict" warnings — expected (it repairs numpy). And don't re-run the model-load cell by itself — it loads a *second* copy and runs the T4 out of memory; re-run only the dial cell to try new coeffs.

In [ ]:
# repeng = the control-vector library. Installing it pins numpy<2, which breaks
# Kaggle's prebuilt PyTorch (compiled against numpy 2.x). The second line repairs
# numpy. The red "dependency conflict" warnings are EXPECTED and harmless.
!pip install -q repeng transformers accelerate bitsandbytes
!pip install -q --force-reinstall "numpy>=2.0,<2.3"

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # faster shard download (preinstalled on Kaggle)

import torch
import numpy as np
if not hasattr(np, "float_"):
    np.float_ = np.float64  # repeng still uses this pre-numpy-2.0 alias; re-add it so import works
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from repeng import ControlVector, ControlModel, DatasetEntry

MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"  # non-gated mirror, no HF login

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token_id = tokenizer.eos_token_id

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")

# Same layer band as the spike (tuned for 8B's 32 layers).
model = ControlModel(model, list(range(-5, -18, -1)))
print("loaded:", MODEL)

## 1. Paste the persona's anchors

This is the only real difference from the spike. Each pair is the **same message twice**: the person's own voice (Hinglish) and a plain-English translation. `repeng` reads the gap between them — and because both sides are *real text from one person*, the direction it learns is that person's voice, not a generic label.

In Vachan these come straight from the capsule (`anchors` ↔ `anchors_english`). Replace the sample below with a real persona's anchors — 8-16 pairs work well.

In [ ]:
# Each pair = the SAME message in the person's own voice (Hinglish) and plain English.
# In Vachan:  positive -> capsule["anchors"][i]["in"]   negative -> capsule["anchors_english"][i]["in"]
# Paste 8-16 real pairs for the persona you want to clone.
ANCHORS = [
    # (their voice / Hinglish,                            plain English translation)
    ("haan bhai bilkul ho jayega, tension mat le",        "Yes, of course it will be done, don't worry."),
    ("scene ye hai ki frontend ka kaam abhi baaki hai",   "The situation is that the frontend work is still pending."),
    ("arre deployment ho gaya, ab testing kar raha hu",   "The deployment is done, now I am running the tests."),
    ("yaar kal milte hain, 10 baje theek hai?",           "Let us meet tomorrow, is 10 o'clock okay?"),
    ("thoda ruk ja, main check karke batata hu",          "Hold on a bit, I will check and let you know."),
    ("haan bhej diya maine, ek baar dekh le",             "Yes, I have sent it, take a look."),
    ("abhi thoda busy hu, baad me baat karein?",          "I am a bit busy right now, can we talk later?"),
    ("ho gaya kaam, kal demo dikha dunga",                "The work is done, I will show the demo tomorrow."),
]
assert len(ANCHORS) >= 4, "too few anchor pairs - the vector needs at least 4 to be stable"

# Frame each anchor as the assistant's reply to a neutral prompt, then read the
# vector at progressive word-truncations (more read positions -> cleaner vector).
USER = "Give me a quick update."
def framed(reply: str, upto: int) -> str:
    head = tokenizer.apply_chat_template(
        [{"role": "user", "content": USER}], tokenize=False, add_generation_prompt=True
    )
    return head + " ".join(reply.split()[:upto])  # slicing past the word count just returns the whole reply

dataset = []
for hinglish, english in ANCHORS:
    # max(), not min(): Hinglish and English sides are rarely the same length, and a
    # word slice past a string's own length is a no-op (not an error) - so using the
    # longer side's count gives the shorter side's full text instead of cutting it off.
    n = max(len(hinglish.split()), len(english.split()))
    for k in range(1, n + 1):
        dataset.append(DatasetEntry(positive=framed(hinglish, k), negative=framed(english, k)))
print(len(dataset), "contrastive pairs from", len(ANCHORS), "persona anchors")
print("--- one positive (their voice) ---")
print(dataset[min(3, len(dataset) - 1)].positive[-120:])

## 2. Extract the persona vector

One pass, no training loop — same cheap extraction as the spike, just over this person's anchors.

In [ ]:
model.reset()
persona_vector = ControlVector.train(model, tokenizer, dataset)

_layers = list(persona_vector.directions.keys())
print("persona vector trained over", len(_layers), "layers")
print("per-layer direction shape:", persona_vector.directions[_layers[0]].shape)

## 3. Turn the dial — toward this person's voice

`coeff > 0` → push toward **their** voice; `coeff < 0` → push toward neutral English; `0` → the untouched model. Because the vector is built from real anchors it is more precise than the generic spike, so a moderate coeff (~+3 to +4) should already read like them. Same 8B caveat applies — push too hard and it garbles.

In [ ]:
def generate(prompt: str, coeff: float) -> str:
    model.reset()
    if coeff != 0:
        model.set_control(persona_vector, coeff)
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    enc = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=80,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    model.reset()
    return tokenizer.decode(out[0, enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

PROMPT = "Can you give me an update on the deployment?"
for c in [-4.0, 0.0, 3.0, 4.0]:
    print(f"\n=========== coeff {c:+} ===========", flush=True)
    print(generate(PROMPT, c), flush=True)

## 4. Read the result + how this plugs into Vachan

If `+3`/`+4` reads recognizably like the person you pasted — closer to *their* phrasing than the generic spike — the per-persona dial works.

**Into the product (not this notebook):**
1. **Pull, don't paste.** Build `ANCHORS` straight from the capsule (`anchors` + `anchors_english` are already stored per persona) — no manual paste.
2. **One vector per persona, cached.** Train once at capsule-build time, store the vector beside the capsule. Generation just loads + sets it.
3. **Serve via Path-B.** Control vectors need *our* forward pass (Groq can't inject), so the steered model runs behind vLLM/transformers on a serverless GPU — only for high-value personas the PFS gate keeps failing.
4. **Tune coeff against the Fidelity Ring.** The neural cosine becomes the objective: pick the coeff that maximizes fidelity to that person's centroid without garbling.

**Honest limit (same as the spike):** base Llama-3.1-8B isn't Hinglish-native, so a heavily code-mixed persona will still garble at high coeff. The fix is a Hinglish-native base (OpenHathi-7B) — deferred.